Instalação do Pydantic

In [ ]:
!pip install pydantic
!pip install pydantic[email]

Importações das bibliotecas necessárias

In [ ]:
import enum
import hashlib
import re
from typing import Any, Self
from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_serializer,
    field_validator,
    model_serializer,
    model_validator,
    SecretStr,
)

Criação de alguns Regex para validar nomes e senhas

In [ ]:
VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")

Criação da Classe Role (Função/Cargo)

In [ ]:
class Role(enum.IntFlag): # Define 5 possíveis funções com seus respectivos valores
    User = 0
    Author = 1
    Editor = 2
    Admin = 4
    SuperAdmin = 8

Criação da Classe User (Usuário)

In [ ]:
class User(BaseModel): # Cria um modelo de dados, usando o BaseModel do Pydantic que valida e estrutura os dados automaticamente
    name: str = Field(examples=["Example"])
    email: EmailStr = Field( # Valida se o valor é um email válido
        examples=["user@arjancodes.com"],
        description="The email address of the user",
        frozen=True, # Congela o campo, isto é depois de criado o email, ele não pode ser alterado
    )
    password: SecretStr = Field( # Esconde o valor do campo
        examples=["Password123"], description="The password of the user", exclude=True # Quando um objeto de User for serializado, a senha não será incluída
    )
    role: Role = Field(
        description="The role of the user",
        examples=[1, 2, 4, 8],
        default=0,
        validate_default=True,
    )

    @field_validator("name") # Usado para adicionar uma validação personalizada para um objeto Pydantic
    def validate_name(cls, v: str) -> str: # v = value
        if not VALID_NAME_REGEX.match(v):  # Verifica se o Regex (definido anteriormente) corresponde ao valor obtido
            raise ValueError(
                "Name is invalid, must contain only letters and be at least 2 characters long"
            )
        return v
    # Field_validator valida campos específicos em um modelo
    @field_validator("role", mode="before") # O mode pode ser "before" - valor ainda não foi definido no objeto - ou "after" - significa que a instância já foi criada e não usa classmethod e sim self
    @classmethod
    def validate_role(cls, v: int | str | Role) -> Role:
        op = {int: lambda x: Role(x), str: lambda x: Role[x], Role: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(
                f"Role is invalid, please use one of the following: {', '.join([x.name for x in Role])}"
            )

    @model_validator(mode="before") # Permite validar todos os dados da instância de uma vez
    @classmethod
    def validate_user_pre(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "name" not in v or "password" not in v: # Verifica se nome ou senha estão ausentes nos dados recebidos
            raise ValueError("Name and password are required")
        if v["name"].casefold() in v["password"].casefold(): # Verifica se a senha contém o nome do usuário
            raise ValueError("Password cannot contain name")
        if not VALID_PASSWORD_REGEX.match(v["password"]): # Verifica se a senha bate com a regex de segurança definida previamente
            raise ValueError(
                "Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number"
            )
        v["password"] = hashlib.sha256(v["password"].encode()).hexdigest() # Criptografa a senha
        return v

    @model_validator(mode="after") # Define um validador que roda depois que o objeto já foi criado
    def validate_user_post(self, v: Any) -> Self: # Garante que somente o "Arjan" possa ser Admin
        if self.role == Role.Admin and self.name != "Arjan":
            raise ValueError("Only Arjan can be an admin")
        return self

    @field_serializer("role", when_used="json") # Define como o campo 'role' deve ser serializado apenas quando convertido para JSON
    @classmethod
    def serialize_role(cls, v) -> str: # Recebe o valor do campo 'role'
        return v.name # Retorna somente o nome do campo 'role' (ex.: Admin)

    @model_serializer(mode="wrap", when_used="json") # Define um serializador personalizado do modelo inteiro, usado apenas para JSON
    def serialize_user(self, serializer, info) -> dict[str, Any]:
        if not info.include and not info.exclude: # Verifica se nenhum campo específico foi pedido ou excluído explicitamente
            return {"name": self.name, "role": self.role.name} # Se for uma serialização simples
        return serializer(self) # Usa o serializador padrão do Pydantic, que respeita o include e exclude


Criação da Função Principal (executa a validação dos dados de exemplo)

In [ ]:
def main() -> None:
    data = {
        "name": "Arjan",
        "email": "example@arjancodes.com",
        "password": "Password123",
        "role": "Admin",
    }
    user = User.model_validate(data) # Faz a validação dos dados do user
    if user:
        print( # Serializa para um dicionário
            "The serializer that returns a dict:",
            user.model_dump(),
            sep="\n",
            end="\n\n",
        )
        print( # Serializa para uma string JSON
            "The serializer that returns a JSON string:",
            user.model_dump(mode="json"),
            sep="\n",
            end="\n\n",
        )
        print( # Também serializa para um JSON, mas agora estamos indicando uma característica do objeto a ser excluída, nesse caso o campo 'role'
            "The serializer that returns a json string, excluding the role:",
            user.model_dump(exclude=["role"], mode="json"),
            sep="\n",
            end="\n\n",
        )
        print("The serializer that encodes all values to a dict:", dict(user), sep="\n") # Serializa para um dicionário todos os valores do usuário até a senha encriptografada


Execução da função Principal

In [ ]:
if __name__ == "__main__":
    main()

The serializer that returns a dict:
{'name': 'Arjan', 'email': 'example@arjancodes.com', 'role': <Role.Admin: 4>}

The serializer that returns a JSON string:
{'name': 'Arjan', 'role': 'Admin'}

The serializer that returns a json string, excluding the role:
{'name': 'Arjan', 'email': 'example@arjancodes.com'}

The serializer that encodes all values to a dict:
{'name': 'Arjan', 'email': 'example@arjancodes.com', 'password': SecretStr('**********'), 'role': <Role.Admin: 4>}
